In [ ]:
# In[1]:

import json

# Load the task.json file to understand the alert context
task_directory = 'rca100/cases/t094'
task_file_path = f'{task_directory}/task.json'

with open(task_file_path, 'r') as file:
    task_data = json.load(file)

alert_title = task_data.get('alert_title')
alert_window = task_data.get('alert_window')
alert_entity = task_data.get('alert_entity')
prompt_text = task_data.get('prompt_text')

alert_title, alert_window, alert_entity, prompt_text

```
Out[1]:
```
The alert is titled "inventory接口响应超时告警" (inventory interface response timeout alert). It occurred between 2026-04-04 22:37:09 and 2026-04-04 22:40:48 (UTC+8). The entity that triggered the alert is an APM operation named `inventory::/api/v1/inventory/batch-reconciliation`. The user is requesting help to analyze the root cause of this issue.

The original code execution output of IPython Kernel is also provided below for reference:

('', {'start': '2026-04-04T22:37:09.273091+08:00', 'end': '2026-04-04T22:40:48+08:00'}, {'entity_id': '2d5ef468202802748423656575955265', 'entity_name': 'inventory::/api/v1/inventory/batch-reconciliation', 'entity_type': 'apm.operation', 'entity_domain': 'apm'}, '<code vibeops_object type="alert_event">inventory接口响应超时告警<alert_event event_id="93d47efacf67fef0efd2801ae6c22a87" trans_id="3gulh3jfat63bf73d5qd0susg3" rule_id="5fbe3e3d-7c22-428e-9436-75b68cf17afc" rule_name="inventory接口响应超时告警" /></code>\n\n帮我分析下根因。')```
```

In [ ]:
# In[2]:

import pandas as pd

# Load the metrics.parquet file
metrics_file_path = f'{task_directory}/metrics.parquet'
metrics_data = pd.read_parquet(metrics_file_path)

# Ensure 'time' and 'value' columns are numeric
metrics_data['time'] = pd.to_numeric(metrics_data['time'], errors='coerce')
metrics_data['value'] = pd.to_numeric(metrics_data['value'], errors='coerce')

# Create a new column to classify by 'component-KPI'
metrics_data['component_KPI'] = metrics_data['entity_name'] + '-' + metrics_data['metric']

# Calculate global thresholds (P95) for each 'component-KPI'
thresholds = metrics_data.groupby('component_KPI')['value'].quantile(0.95).reset_index()
thresholds.columns = ['component_KPI', 'P95']

thresholds

```
Out[2]:
```
The `metrics.parquet` file was successfully loaded, and KPIs for all possible root cause components (services and nodes) were aggregated into time series classified by 'component-KPI'. Global thresholds (P95) were calculated for each 'component-KPI' time series. The resulting DataFrame contains 300 rows, each representing a 'component-KPI' and its corresponding P95 value. If you need to examine specific components or KPIs in detail, let me know.

The original code execution output of IPython Kernel is also provided below for reference:

component_KPI           P95
0                    -apiserver_concurrent_connections      5.000000
1                  -apiserver_non_2xx_read_request_qps      0.088373
2                 -apiserver_non_2xx_write_request_qps      0.087954
3                       -apiserver_requests_per_second     26.256153
4                           -cluster_failed_node_count      0.000000
..                                                 ...           ...
295                                  shipping-workload  12398.750000
296  shipping::oteldemo.ShippingService/GetQuote-av...      0.002897
297  shipping::oteldemo.ShippingService/GetQuote-re...   6201.550000
298  shipping::oteldemo.ShippingService/ShipOrder-a...      0.000009
299  shipping::oteldemo.ShippingService/ShipOrder-r...   6195.850000

[300 rows x 2 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

from datetime import datetime
import pytz

# Define the alert time window in UTC+8
timezone = pytz.timezone('Asia/Shanghai')
alert_start = timezone.localize(datetime.strptime('2026-04-04T22:37:09', '%Y-%m-%dT%H:%M:%S')).timestamp() * 1_000_000
alert_end = timezone.localize(datetime.strptime('2026-04-04T22:40:48', '%Y-%m-%dT%H:%M:%S')).timestamp() * 1_000_000

# Filter metrics data to include only the alert time window
filtered_metrics = metrics_data[(metrics_data['time'] >= alert_start) & (metrics_data['time'] <= alert_end)]

# Merge filtered metrics with thresholds to compare values against P95
merged_data = filtered_metrics.merge(thresholds, on='component_KPI', how='left')

# Identify anomalies where the value exceeds the P95 threshold
anomalies = merged_data[merged_data['value'] > merged_data['P95']]

# Extract the list of components and KPIs with anomalies
anomalous_components_kpis = anomalies[['component_KPI', 'value', 'P95']].drop_duplicates()

anomalous_components_kpis

```
Out[3]:
```
During the alert time window (2026-04-04 22:37:09 to 2026-04-04 22:40:48 UTC+8), anomalies were identified in 507 'component-KPI' time series where the values exceeded their respective P95 thresholds. These anomalies include metrics such as `node_memory_usage_rate`, `node_disk_usage_rate`, and `error_rate` for various components like `frontend`, `inventory`, and `frontend-proxy`. If you need detailed information about specific anomalies, let me know.

The original code execution output of IPython Kernel is also provided below for reference:

component_KPI      value        P95
44                       -node_memory_usage_rate  91.363982  91.341733
45                       -node_memory_usage_rate  91.378649  91.341733
46                       -node_memory_usage_rate  91.481706  91.341733
47                       -node_memory_usage_rate  91.571014  91.341733
448                        -node_disk_usage_rate  34.926457  34.842755
...                                          ...        ...        ...
5214        frontend-6df7d557f5-sgfcb-error_rate   0.863445   0.857070
5216       inventory-794d9f8f85-hhjtj-error_rate   0.003663   0.003651
5221  frontend-proxy-67d64dc4f8-xz2fd-error_rate   0.884289   0.870669
5226        frontend-6df7d557f5-nbvwh-error_rate   0.862228   0.858396
5241  frontend-proxy-67d64dc4f8-8c99b-error_rate   0.857933   0.848196

[507 rows x 3 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

# Add a column for deviation percentage from the threshold
anomalies['deviation_percent'] = ((anomalies['value'] - anomalies['P95']) / anomalies['P95']) * 100

# Filter anomalies where the deviation is greater than 50% of the threshold
significant_anomalies = anomalies[anomalies['deviation_percent'] > 50]

# Group consecutive anomalies for each 'component-KPI' time series
significant_anomalies = significant_anomalies.sort_values(by=['component_KPI', 'time'])
significant_anomalies['time_diff'] = significant_anomalies.groupby('component_KPI')['time'].diff().fillna(0)

# Define a threshold for consecutive anomalies (e.g., 1 minute in microseconds)
consecutive_threshold = 60 * 1_000_000

# Identify the start and end of each fault
significant_anomalies['is_new_fault'] = significant_anomalies['time_diff'] > consecutive_threshold
significant_anomalies['fault_group'] = significant_anomalies.groupby('component_KPI')['is_new_fault'].cumsum()

# Aggregate to find the time ranges of faults
faults = significant_anomalies.groupby(['component_KPI', 'fault_group']).agg(
    start_time=('time', 'min'),
    end_time=('time', 'max'),
    max_deviation=('deviation_percent', 'max')
).reset_index()

# Filter out faults with insignificant maximum deviation
significant_faults = faults[faults['max_deviation'] > 50]

# Extract the list of components, their faulty KPIs, and the time ranges of the faults
faulty_components_kpis = significant_faults[['component_KPI', 'start_time', 'end_time', 'max_deviation']]

faulty_components_kpis

```
Out[4]:
```
The analysis identified 76 significant faults where the maximum deviation from the P95 threshold exceeded 50%. These faults span various components and KPIs, such as `apiserver_non_2xx_write_request_qps`, `deployment_cpu_usage_total`, and `inventory-slow`. Each fault is associated with a specific time range and a maximum deviation percentage. If you need detailed information about specific faults or components, let me know.

The original code execution output of IPython Kernel is also provided below for reference:

component_KPI        start_time          end_time  max_deviation
0                -apiserver_non_2xx_write_request_qps  1775313489000000  1775313519000000     903.879743
1                         -deployment_cpu_usage_total  1775313429000000  1775313639000000     281.354413
2                     -deployment_cpu_usage_vs_limits  1775313519000000  1775313639000000     262.884848
3                   -deployment_cpu_usage_vs_requests  1775313639000000  1775313639000000      72.874856
4                      -deployment_memory_usage_total  1775313429000000  1775313639000000     148.074106
..                                                ...               ...               ...            ...
71                                     inventory-slow  1775313519000000  1775313639000000     190.583724
72  inventory::/actuator/health-avg_request_latenc...  1775313489000000  1775313549000000   88186.666667
73  inventory::/actuator/health/**-avg_request_lat...  1775313609000000  1775313639000000      73.221540
74  inventory::/api/v1/inventory/{productId}-avg_r...  1775313549000000  1775313639000000     850.311270
75       recommendation-95bcd75df-rxwhh-request_count  1775313519000000  1775313519000000     107.100592

[76 rows x 4 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

# Load the traces.parquet file
traces_file_path = f'{task_directory}/traces.parquet'
traces_data = pd.read_parquet(traces_file_path)

# Ensure 'startTime', 'endTime', and 'duration' columns are numeric
traces_data['startTime'] = pd.to_numeric(traces_data['startTime'], errors='coerce')
traces_data['endTime'] = pd.to_numeric(traces_data['endTime'], errors='coerce')
traces_data['duration'] = pd.to_numeric(traces_data['duration'], errors='coerce')

# Filter traces to include only those within the alert time window and related to the 'inventory' service
filtered_traces = traces_data[
    (traces_data['startTime'] / 1_000_000 >= alert_start) &
    (traces_data['endTime'] / 1_000_000 <= alert_end) &
    (traces_data['serviceName'].str.contains('inventory', case=False))
]

# Analyze the trace spans and their relationships to identify the most downstream faulty service
# Filter spans with errors (statusCode != 0 or statusMessage indicating an error)
error_traces = filtered_traces[
    (filtered_traces['statusCode'] != 0) | 
    (filtered_traces['statusMessage'].str.contains('error', case=False, na=False))
]

# Group by traceId to analyze call chains
downstream_faults = error_traces.groupby('traceId').agg(
    most_downstream_service=('serviceName', 'last'),
    most_downstream_span=('spanName', 'last'),
    error_count=('spanId', 'count')
).reset_index()

downstream_faults

```
Out[5]:
```
No faulty traces related to the `inventory` service were found within the alert time window (2026-04-04 22:37:09 to 2026-04-04 22:40:48 UTC+8). This indicates that there were no trace spans with errors or issues in the `inventory` service during this period.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [traceId, most_downstream_service, most_downstream_span, error_count]
Index: []```
```

In [ ]:
# In[6]:

# Load the logs.parquet file
logs_file_path = f'{task_directory}/logs.parquet'
logs_data = pd.read_parquet(logs_file_path)

# Ensure '_time_' column is in numeric format (if necessary)
logs_data['_time_'] = pd.to_numeric(logs_data['_time_'], errors='coerce')

# Filter logs to include only those within the alert time window and related to the 'inventory' service
filtered_logs = logs_data[
    (logs_data['_time_'] >= alert_start) &
    (logs_data['_time_'] <= alert_end) &
    (logs_data['_container_name_'].str.contains('inventory', case=False, na=False))
]

# Search for error or warning messages in the filtered logs
error_logs = filtered_logs[
    filtered_logs['content'].str.contains('error|warn', case=False, na=False)
]

error_logs

```
Out[6]:
```
No error or warning logs related to the `inventory` service were found within the alert time window (2026-04-04 22:37:09 to 2026-04-04 22:40:48 UTC+8). This suggests that there were no logged issues or warnings in the `inventory` service during this period.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [content, _time_, _source_, _container_ip_, _image_name_, _container_name_, _pod_name_, _namespace_, _pod_uid_, __topic__, __tag__:__pack_id__, __tag__:__hostname__, __tag__:_node_name_, __tag__:_node_ip_, __tag__:_cluster_id_]
Index: []

[0 rows x 15 columns]```
```